### 01 — Dataset Exploration and Preparation

What this notebook does:

- define the project data paths
- load and inspect the dataset annotations
- review the available clinical, image, and diagnostic fields
- identify missing or unavailable values
- verify that ultrasound images and tumour masks are present
- exclude normal cases so that the classification task is benign vs malignant
- separate the available variables according to their intended role
- encode the classification target
- create reproducible training, validation, and test splits
- save the processed splits for reuse by the later notebooks.

The saved splits are reused throughout the project so that segmentation, classification, multimodal fusion, and report generation are trained and evaluated on the same cases.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1. Define Project Paths

- `data/raw/labels.xlsx` contains the dataset annotations.
- `data/raw/images/` contains the ultrasound images and segmentation masks.
- `data/processed/` is used to store the train, validation, and test CSV files generated later in this notebook.


In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

LABEL_FILE = RAW_DIR / "labels.xlsx"
IMAGE_DIR = RAW_DIR / "images"

### 2. Load and Inspect the Clinical Dataset

The annotation spreadsheet is loaded into a Pandas DataFrame.


In [3]:
df = pd.read_excel(LABEL_FILE)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (256, 21)


,CaseID,Image_filename,Mask_tumor_filename,Mask_other_filename,Pixel_size,Age,Tissue_composition,Signs,Symptoms,Shape,...,Echogenicity,Posterior_features,Halo,Calcifications,Skin_thickening,Interpretation,BIRADS,Verification,Diagnosis,Classification
0,1,case001.png,case001_tumor.png,NaN,0.007812,57,heterogeneous: predominantly fat,breast scar,family history of breast/ovarian cancer,irregular,...,heterogeneous,shadowing,no,no,yes,Breast scar (surgery)&Breast scar (radiotherapy),2,confirmed by follow-up care,not applicable,benign
1,2,case002.png,case002_tumor.png,NaN,0.006462,not available,homogeneous: fat,not available,not available,oval,...,hypoechoic,no,no,no,no,Dysplasia&Fibroadenoma,4b,confirmed by biopsy,Intramammary lymph node,benign
2,3,case003.png,case003_tumor.png,NaN,0.006944,56,heterogeneous: predominantly fat,no,nipple discharge,oval,...,hyperechoic,no,no,no,no,Duct filled with thick fluid&Intraductal papil...,4a,confirmed by biopsy,Usual ductal hyperplasia (UDH)&Pseudoangiomato...,benign
3,4,case004.png,case004_tumor.png,NaN,0.007812,43,homogeneous: fibroglandular,no,no,round,...,hypoechoic,no,no,no,no,Cyst filled with thick fluid,3,confirmed by follow-up care,not applicable,benign
4,5,case005.png,case005_tumor.png,NaN,0.007812,67,homogeneous: fat,nipple retraction&palpable,family history of breast/ovarian cancer,oval,...,complex cystic/solid,enhancement,no,no,no,Suspicion of malignancy&Intraductal papilloma,4b,confirmed by biopsy,Encapsulated papillary carcinoma&Ductal carcin...,malignant


In [4]:
print(df.columns.tolist())

['CaseID', 'Image_filename', 'Mask_tumor_filename', 'Mask_other_filename', 'Pixel_size', 'Age', 'Tissue_composition', 'Signs', 'Symptoms', 'Shape', 'Margin', 'Echogenicity', 'Posterior_features', 'Halo', 'Calcifications', 'Skin_thickening', 'Interpretation', 'BIRADS', 'Verification', 'Diagnosis', 'Classification']


In [5]:
important_columns = [
    "CaseID",
    "Image_filename",
    "Mask_tumor_filename",
    "Age",
    "Tissue_composition",
    "Signs",
    "Symptoms",
    "Shape",
    "Margin",
    "Echogenicity",
    "Posterior_features",
    "Halo",
    "Calcifications",
    "Skin_thickening",
    "Interpretation",
    "BIRADS",
    "Verification",
    "Diagnosis",
    "Classification"
]

display(df[important_columns].head())

,CaseID,Image_filename,Mask_tumor_filename,Age,Tissue_composition,Signs,Symptoms,Shape,Margin,Echogenicity,Posterior_features,Halo,Calcifications,Skin_thickening,Interpretation,BIRADS,Verification,Diagnosis,Classification
0,1,case001.png,case001_tumor.png,57,heterogeneous: predominantly fat,breast scar,family history of breast/ovarian cancer,irregular,not circumscribed - indistinct,heterogeneous,shadowing,no,no,yes,Breast scar (surgery)&Breast scar (radiotherapy),2,confirmed by follow-up care,not applicable,benign
1,2,case002.png,case002_tumor.png,not available,homogeneous: fat,not available,not available,oval,not circumscribed - indistinct,hypoechoic,no,no,no,no,Dysplasia&Fibroadenoma,4b,confirmed by biopsy,Intramammary lymph node,benign
2,3,case003.png,case003_tumor.png,56,heterogeneous: predominantly fat,no,nipple discharge,oval,circumscribed,hyperechoic,no,no,no,no,Duct filled with thick fluid&Intraductal papil...,4a,confirmed by biopsy,Usual ductal hyperplasia (UDH)&Pseudoangiomato...,benign
3,4,case004.png,case004_tumor.png,43,homogeneous: fibroglandular,no,no,round,circumscribed,hypoechoic,no,no,no,no,Cyst filled with thick fluid,3,confirmed by follow-up care,not applicable,benign
4,5,case005.png,case005_tumor.png,67,homogeneous: fat,nipple retraction&palpable,family history of breast/ovarian cancer,oval,circumscribed,complex cystic/solid,enhancement,no,no,no,Suspicion of malignancy&Intraductal papilloma,4b,confirmed by biopsy,Encapsulated papillary carcinoma&Ductal carcin...,malignant


In [6]:
print(df["Classification"].value_counts(dropna=False))

Classification
benign       154
malignant     98
normal         4
Name: count, dtype: int64


### 3. Check Missing or Unavailable Data

The BrEaST spreadsheet does not store every missing clinical value as a standard value. Some entries are stored as text such as 'not available' or 'not applicable'

The following check searches explicitly for these values in addition to blank string. to show a overview of incomplete clinical information


In [7]:
missing_strings = ["not available", "not applicable", ""]

for col in df.columns:
    count = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(missing_strings)
        .sum()
    )

    if count > 0:
        print(f"{col}: {count}")

Age: 42
Tissue_composition: 49
Signs: 46
Symptoms: 98
Shape: 4
Margin: 4
Echogenicity: 4
Posterior_features: 4
Halo: 4
Calcifications: 4
Interpretation: 4
Verification: 4
Diagnosis: 59


### 4. Verify Image and Tumour-Mask Files

The notebook checks whether the filenames listed in the spreadsheet correspond to files that are actually present in the image directory.

This helps identify broken paths or missing files


In [8]:
def file_exists(filename):
    if pd.isna(filename):
        return False

    return (IMAGE_DIR / str(filename)).exists()


df["image_exists"] = df["Image_filename"].apply(file_exists)

print(df["image_exists"].value_counts())

image_exists
True    256
Name: count, dtype: int64


In [9]:
def all_masks_exist(mask_value):
    if pd.isna(mask_value):
        return False

    filenames = str(mask_value).split("&")

    return all(
        (IMAGE_DIR / filename.strip()).exists()
        for filename in filenames
    )


df["tumor_mask_exists"] = df["Mask_tumor_filename"].apply(all_masks_exist)

print(df["tumor_mask_exists"].value_counts())

tumor_mask_exists
True     252
False      4
Name: count, dtype: int64


### 5. Remove Normal Cases

The original dataset has a small number of normal examinations. These cases are removed from the modelling subset because the project's main classification task is benign and malignant

Only records whose `Classification` value is either `benign` or `malignant` are retained in `df_model`.


In [10]:
df_model = df[
    df["Classification"]
    .astype(str)
    .str.lower()
    .isin(["benign", "malignant"])
].copy()

print(df_model.shape)
print(df_model["Classification"].value_counts())

(252, 23)
Classification
benign       154
malignant     98
Name: count, dtype: int64


### 6. Group Variables by Their Intended Role

#### Patient clinical variables
These are the structured clinical inputs used by the ClinicalBERT branch:
- Age
- Tissue composition
- Signs
- Symptoms

#### Image descriptor variables
These describe lesion appearance in the ultrasound image, such as shape, margin, and echogenicity. They are used for analysis and report construction, but they are not used as direct classification inputs in the main multimodal model.

#### Diagnostic variables
These contain assessments or outcomes such as BI-RADS, diagnosis, verification, and final classification. They are kept separate because directly using them as classifier inputs could introduce target leakage.


In [11]:
patient_clinical_columns = [
    "Age",
    "Tissue_composition",
    "Signs",
    "Symptoms"
]

image_descriptor_columns = [
    "Shape",
    "Margin",
    "Echogenicity",
    "Posterior_features",
    "Halo",
    "Calcifications",
    "Skin_thickening"
]

diagnostic_columns = [
    "Interpretation",
    "BIRADS",
    "Verification",
    "Diagnosis",
    "Classification"
]

### 7. Encode the Binary Classification Target

The text labels are converted into numerical values for machine-learning use:
- benign → 0
- malignant → 1

The encoded value is stored in a new target column.


In [20]:
df_model["target"] = (
    df_model["Classification"]
    .str.lower()
    .map({
        "benign": 0,
        "malignant": 1
    })
)

print(df_model[["Classification", "target"]].tail())

    Classification  target
251         benign       0
252         benign       0
253      malignant       1
254      malignant       1
255         benign       0


### 8. Create Training, Validation, and Test Splits

The dataset is divided into approximately:
- 70% training
- 15% validation
- 15% testing

The split is performed in two stages using train_test_split.
stratify=df_model["target"] preserves the benign/malignant class ratio across the subsets.
random_state=42 makes the split reproducible.

These splits are saved and reused in later notebooks so that all model components are trained and evaluated on a consistent partition of the data.


In [13]:
from sklearn.model_selection import train_test_split

In [14]:
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.30,
    stratify=df_model["target"],
    random_state=42
)

In [15]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

In [16]:
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 176
Validation: 38
Testing: 38


In [17]:
for name, split in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(split["Classification"].value_counts())


Train
Classification
benign       108
malignant     68
Name: count, dtype: int64

Validation
Classification
benign       23
malignant    15
Name: count, dtype: int64

Test
Classification
benign       23
malignant    15
Name: count, dtype: int64


## 9. Save the Processed Dataset Splits

The three subsets are written to data/processed/ as:
- train.csv
- val.csv
- test.csv

These are the dataset partitions used by the segmentation, image-encoding, multimodal classification, report-generation, and final-evaluation notebooks.

In [18]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)